
# # 📚 RAG-System mit LangChain, ChromaDB und Gemini 2
# Dieses Notebook implementiert ein einfaches Retrieval-Augmented Generation (RAG) System.
# Es verwendet ChromaDB zur Dokumentenspeicherung, LangChain für Workflow-Management und das Modell `gemini-2.0-flash`.


# 📥 Bibliotheken importieren

In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain.embeddings import GooglePalmEmbeddings
from langchain.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain

import os

from langchain_community.embeddings import HuggingFaceEmbeddings


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"

# ## 🌐 Schritt 1: Wikipedia-Seite laden

In [2]:
url = "https://en.wikipedia.org/wiki/2025_in_science"
loader = WebBaseLoader(url)
documents = loader.load()

# ## ✂️ Schritt 2: Text in Chunks aufteilen

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)
print(f"Anzahl der Chunks: {len(chunks)}")

Anzahl der Chunks: 90


# ## 🧠 Schritt 3: Vektorisierung und Speicherung in ChromaDB

In [4]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\volodymyr\AppData\Local\Temp\ipykernel_12252\4093584008.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
c:\Users\volodymyr\AppData\Local\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
 

persist_directory = "chroma_db"
if os.path.exists(persist_directory) and os.listdir(persist_directory):
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        persist_directory=persist_directory
    )
    vectorstore.persist()


C:\Users\volodymyr\AppData\Local\Temp\ipykernel_12252\1983505030.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [6]:
os.environ["LANGCHAIN_PROJECT"] = "RAG Wikipedia 2025"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
from langchain.callbacks.tracers import LangChainTracer

#from langchain.callbacks.manager import CallbackManager
tracer = LangChainTracer(project_name="RAG Wikipedia 2025")



# ## 💬 Schritt 4: Dialogsystem mit Gedächtnis (Memory)

## 💾 Kontextverwaltung über mehrere Sitzungen
Wir verwenden `FileChatMessageHistory`, um den Gesprächsverlauf zwischen Sitzungen zu speichern.
Dies ermöglicht eine langfristige Konversationshistorie.

In [7]:
from langchain.memory.chat_message_histories import FileChatMessageHistory
from langchain.retrievers.multi_query import MultiQueryRetriever
message_history = FileChatMessageHistory("chat_history.json")

memory = ConversationBufferMemory(
    memory_key="chat_history",
    chat_memory=message_history,
    return_messages=True
)






llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=multi_query_retriever,
    memory=memory, 
    callbacks=[tracer]
)

C:\Users\volodymyr\AppData\Local\Temp\ipykernel_12252\1027373561.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


Prompt erstellen

In [8]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

assert len(example_messages) == 1
print(example_messages[0].content)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


# ## 🧪 Schritt 5: Beispiel-Dialog

ConversationalRetrievalChain-basierte RAG mit ConversationBufferMemory

In [9]:
input_question = "When did the second Trump administration impose an immediate freeze on research grants, communications, hiring, and meetings at the National Institutes of Health?"

rag_chain.invoke(input_question)

{'question': 'When did the second Trump administration impose an immediate freeze on research grants, communications, hiring, and meetings at the National Institutes of Health?',
 'chat_history': [HumanMessage(content='When did the second Trump administration impose an immediate freeze on research grants, communications, hiring, and meetings at the National Institutes of Health?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='According to the provided text, the second Trump administration imposed an immediate freeze on scientific grants, communications, hiring, and meetings at the National Institutes of Health (NIH) on January 22, 2025.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content="How much of the institute's operations were affected by this decision?", additional_kwargs={}, response_metadata={}),
  AIMessage(content='$47.4 billion worth of activities at the National Institutes of Health (NIH) were affected.', additional_kwargs={}, response_

In [10]:

fragen = [
    "When did the second Trump administration impose an immediate freeze on research grants, communications, hiring, and meetings at the National Institutes of Health?",
    "How much of the institute's operations were affected by this decision?",
    "When did astronomers report the discovery of Saturn's new moons?",
    "Which countries' telescopes were used? How many new moons of Saturn were discovered?",
    "What is the total number of confirmed satellites of Saturn currently known?" 
]

for frage in fragen:
   
    antwort = rag_chain.run(frage)
    print(f"\n🙋 Frage: {frage}\n🤖 Antwort: {antwort}")
    

C:\Users\volodymyr\AppData\Local\Temp\ipykernel_12252\606232336.py:11: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  antwort = rag_chain.run(frage)



🙋 Frage: When did the second Trump administration impose an immediate freeze on research grants, communications, hiring, and meetings at the National Institutes of Health?
🤖 Antwort: Die zweite Trump-Administration verhängte am 22. Januar 2025 einen sofortigen Stopp für wissenschaftliche Zuschüsse, Kommunikationen, Einstellungen und Treffen an den National Institutes of Health (NIH).

🙋 Frage: How much of the institute's operations were affected by this decision?
🤖 Antwort: Ich bin mir nicht sicher, wie ich das beantworten soll. Die bereitgestellten Informationen enthalten keine Informationen über die Aktivitäten des Instituts.

🙋 Frage: When did astronomers report the discovery of Saturn's new moons?
🤖 Antwort: Am 11. März 2025 berichteten Astronomen über die Entdeckung von 128 neuen Monden des Saturn.

🙋 Frage: Which countries' telescopes were used? How many new moons of Saturn were discovered?
🤖 Antwort: Das Canada-France-Hawaii-Teleskop wurde verwendet und 128 neue Saturnmonde wur